# REINFORCE and the Policy Gradient Theorem — interactive companion

This notebook is the **interactive companion** to
[Post 2a: REINFORCE and the Policy Gradient Theorem](../posts/02a-reinforce-and-policy-gradient.qmd).
The post does the full derivation; this notebook lets you *poke at* the
algorithm.

Read the post first if you want the math from first principles. Open this
notebook if you want to change `lr=0.05` to `lr=1.0` and see the policy
implode in real time.

**What you'll do here (≈ 25 minutes):**
1. Visualize the log-derivative trick on a 1D continuous bandit.
2. Implement REINFORCE in ~15 lines of NumPy.
3. Train it on a gridworld; toggle baselines and watch variance shrink.
4. Extend to continuous actions with a learnable Gaussian policy.
5. Run the suggested experiments and break things on purpose.

**Prerequisites.** Clone the repo and `pip install -e ".[dev]"`. This
notebook imports from `nano_agents.policy_gradient` for convenience, but
all the core algorithms are also reimplemented inline so you can see exactly
what's happening.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Make plots a reasonable default size.
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["figure.dpi"] = 110

# The shared library that the blog posts use. We'll reimplement key pieces
# below, but importing also gives us the gridworld and the trained value
# iteration baseline to compare against.
from nano_agents.mdp import TwoGoalGridWorld, value_iteration, simulate_step
from nano_agents.policy_gradient import (
    SoftmaxPolicy, GaussianPolicy,
    collect_trajectory, compute_returns,
    train_reinforce,
)

rng = np.random.default_rng(0)

## 1. The log-derivative trick, made visual

The whole policy-gradient apparatus rests on a single identity:

$$
\nabla_\theta \mathbb{E}_{a \sim \pi_\theta}[r(a)] \;=\; \mathbb{E}_{a \sim \pi_\theta}\big[r(a)\,\nabla_\theta \log \pi_\theta(a)\big].
$$

The LHS is a gradient we don't know how to compute (the distribution depends
on $\theta$). The RHS is an expectation we *can* estimate from samples.

To see this concretely, take a 1D Gaussian policy $\pi(a;\mu) = \mathcal{N}(\mu, 1^2)$
and a reward $r(a) = 1 - (a-2)^2$. The optimum is $\mu^\star = 2$.

In [ ]:
# Simplest possible policy: 1D Gaussian with learnable mean.
mu = 0.0      # start far from optimum
sigma = 1.0

def reward(a):
    return 1.0 - (a - 2.0) ** 2

# Visualize three things:
#   - the policy density at current mu
#   - the reward function
#   - the integrand of the policy gradient (this is what REINFORCE averages)
a_grid = np.linspace(-3, 5, 500)
pi_a = np.exp(-0.5 * ((a_grid - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))
r_a = reward(a_grid)
score = (a_grid - mu) / sigma ** 2   # d/dmu log pi(a; mu)
integrand = pi_a * r_a * score        # the thing we sample-average

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(a_grid, pi_a, label=f"$\\pi(a;\\mu={mu})$", color="C0")
ax2 = axes[0].twinx()
ax2.plot(a_grid, r_a, color="C3", label="$r(a)$")
axes[0].set_xlabel("action $a$"); axes[0].set_title("Policy and reward")
axes[0].legend(loc="upper left"); ax2.legend(loc="upper right")

axes[1].plot(a_grid, integrand, color="black")
axes[1].fill_between(a_grid, integrand, where=integrand > 0, color="green", alpha=0.3,
                     label="positive contribution")
axes[1].fill_between(a_grid, integrand, where=integrand < 0, color="red", alpha=0.3,
                     label="negative contribution")
grad = float(np.trapezoid(integrand, a_grid))
axes[1].set_title(f"Gradient integrand. $\\nabla_\\mu J \\approx {grad:+.3f}$")
axes[1].set_xlabel("action $a$"); axes[1].legend()
plt.tight_layout(); plt.show()

### Try this

The gradient above is positive (~+3.9), so REINFORCE will increase $\mu$ —
move the policy toward the reward peak. Now try modifying the cell above:

- Set `mu = 2.0`. What's the gradient now? *(Should be near zero — we're at the optimum.)*
- Set `mu = 5.0`. Which side of zero does the gradient land on?
- Change `reward` to `lambda a: -np.abs(a - 2)`. Does the picture change qualitatively?
- Make `sigma = 0.3`. The policy gets sharper. What happens to the gradient
  magnitude when $\mu$ is far from the peak?

## 2. REINFORCE in 15 lines

The same gradient identity, applied to trajectories of an MDP, gives the
**policy gradient theorem**:

$$
\nabla_\theta J(\theta) \;=\; \mathbb{E}_{\tau \sim \pi_\theta}\!\Big[\sum_t G_t \,\nabla_\theta \log \pi_\theta(a_t \mid s_t)\Big].
$$

For a softmax policy $\pi_\theta(a\mid s) = \mathrm{softmax}(\theta_{s,\cdot})_a$,
the score $\nabla_\theta \log \pi$ has a closed form, and the whole REINFORCE
update fits in a few lines.

Here it is, reimplemented inline so you can see the entire algorithm at once:

In [ ]:
class TinyPolicy:
    '''Softmax policy with tabular logits theta[state, action].'''
    def __init__(self, nS, nA):
        self.theta = np.zeros((nS, nA))
    def probs(self, s):
        z = self.theta[s] - self.theta[s].max()
        e = np.exp(z); return e / e.sum()
    def sample(self, s, rng):
        return int(rng.choice(len(self.theta[s]), p=self.probs(s)))

def reinforce_update(policy, traj, returns, lr, baseline=0.0):
    '''Apply ONE REINFORCE gradient step from a single trajectory.

    traj: list of (state_idx, action_idx, reward)
    returns: precomputed G_t values
    baseline: scalar or per-state array subtracted from G_t for variance reduction
    '''
    for t, (s, a, _) in enumerate(traj):
        A = returns[t] - (baseline[s] if hasattr(baseline, "__len__") else baseline)
        p = policy.probs(s)
        # The softmax score: nabla log pi(a|s) = e_a - p, only nonzero in row s.
        policy.theta[s] += lr * A * (-p)
        policy.theta[s, a] += lr * A

print("That's it. Two functions. Now let's run it.")

## 3. Train on a gridworld

Same TwoGoalGridWorld as Post 2a: a 5×5 grid with a small +1 terminal and a
big +10 terminal, with walls and a step penalty. Optimal policy heads toward
the +10.

In [ ]:
env = TwoGoalGridWorld(slip=0.0, step_reward=-0.04)
gamma = 0.95

# For reference: what does value iteration say?
V_star, pi_star, _ = value_iteration(env, gamma=gamma)
print(f"Optimal V at start state: {V_star[env.state_to_idx[env.start]]:.3f}")
print(f"Number of states: {env.nS}, actions: {env.nA}")

In [ ]:
# Train vanilla REINFORCE.
policy = TinyPolicy(env.nS, env.nA)
rng = np.random.default_rng(0)
n_episodes = 1500
returns_history = np.zeros(n_episodes)

non_terminal = [s for s in env.states if not env.is_terminal(s)]
for ep in range(n_episodes):
    start = non_terminal[rng.integers(len(non_terminal))]
    traj = collect_trajectory(env, policy, start, rng, max_steps=200)
    rewards = [r for (_, _, r) in traj]
    G = compute_returns(rewards, gamma)
    reinforce_update(policy, traj, G, lr=0.05)  # no baseline yet
    returns_history[ep] = sum(rewards)

# Plot a moving average.
w = 50
ma = np.convolve(returns_history, np.ones(w)/w, mode="valid")
plt.plot(ma, label="REINFORCE (no baseline)")
plt.axhline(V_star[env.state_to_idx[env.start]], color="black", linestyle="--",
            label="$V^\\star$ from start", alpha=0.5)
plt.xlabel("episode"); plt.ylabel(f"return ({w}-ep moving average)")
plt.legend(); plt.grid(alpha=0.3); plt.show()

You should see returns climb from ~5 to ~9.8 by episode ~500, then plateau.

### Try this

Make REINFORCE break:
- Change `lr=0.05` to `lr=2.0`. What happens?
- Change `lr=0.05` to `lr=0.001`. Does it still converge?
- Drop `n_episodes` to `200`. Is the policy converged?

Each of these isolates a way vanilla REINFORCE fails: large lr is unstable,
small lr is glacial, short training is unconverged. PPO (covered in Post 2c)
fixes the first one.

## 4. Baselines: the cheapest variance reduction trick

The classical observation: subtracting any state-only function $b(s)$ from
the return inside the gradient is **unbiased** (you can prove this in one line)
but can dramatically reduce variance.

Let's compare three baselines on the same gridworld:
- No baseline.
- Running mean of episode returns.
- $V^\star$ (oracle baseline — we cheat using value iteration).

In [ ]:
def train_with_baseline(env, n_episodes, lr, gamma, baseline_kind, V_star=None):
    policy = TinyPolicy(env.nS, env.nA)
    rng = np.random.default_rng(0)
    returns = np.zeros(n_episodes)
    running = 0.0; n = 0
    non_terminal = [s for s in env.states if not env.is_terminal(s)]
    for ep in range(n_episodes):
        start = non_terminal[rng.integers(len(non_terminal))]
        traj = collect_trajectory(env, policy, start, rng, max_steps=200)
        rewards = [r for (_, _, r) in traj]
        G = compute_returns(rewards, gamma)
        # Pick baseline.
        if baseline_kind == "none":
            b = 0.0
        elif baseline_kind == "mean":
            b = running
        elif baseline_kind == "Vstar":
            b = V_star
        reinforce_update(policy, traj, G, lr=lr, baseline=b)
        ep_R = sum(rewards); returns[ep] = ep_R
        n += 1; running += (ep_R - running) / n
    return returns

results = {}
for name in ["none", "mean", "Vstar"]:
    results[name] = train_with_baseline(env, 1500, lr=0.05, gamma=gamma,
                                          baseline_kind=name, V_star=V_star)

w = 50
for name, ret in results.items():
    ma = np.convolve(ret, np.ones(w)/w, mode="valid")
    plt.plot(ma, label=f"baseline = {name}")
plt.xlabel("episode"); plt.ylabel(f"return ({w}-ep MA)")
plt.legend(); plt.grid(alpha=0.3); plt.show()

**Surprising honest result** (also discussed in Post 2a §5):

The $V^\star$ "oracle" baseline does **not** uniformly help. Early in
training, the policy is far from optimal, so $G_t - V^\star(s_t)$ is mostly
*very negative* — the agent gets pushed away from everything, including good
actions. The running mean tracks the *current* policy's value more
accurately and helps more.

The right baseline is $V^{\pi_\theta}$ for the *current* policy — which is
what actor-critic methods learn online. That's the next post.

### Try this
- Set `lr=0.5` and compare the three again. With more aggressive updates,
  the no-baseline case becomes much noisier — does the running-mean
  baseline help more relatively?
- Add a fourth baseline: a fixed scalar like `b = 5.0`. Does it work?
  *(It should — any state-independent baseline is unbiased.)*

## 5. Continuous action space

Q-learning needs $\arg\max_a Q(s, a)$ — easy for finite actions, hard for
continuous ones. Policy gradient doesn't care: just parameterize a Gaussian
$\pi(a; \mu, \sigma)$ and take gradients w.r.t. $\mu$ and $\log\sigma$.

Reward function: $r(a) = \exp(-(a-2)^2)$. Optimum at $a=2$ with $r=1$.

In [ ]:
class LearnableGaussian:
    '''Single-state Gaussian policy with learnable mu, log_sigma.'''
    def __init__(self, mu_init=0.0, log_sigma_init=0.5):
        self.mu = mu_init
        self.log_sigma = log_sigma_init
    @property
    def sigma(self):
        return float(np.exp(self.log_sigma))
    def sample(self, rng):
        return float(rng.normal(self.mu, self.sigma))
    def grads(self, a):
        s2 = self.sigma ** 2
        return (a - self.mu) / s2, ((a - self.mu) ** 2 / s2) - 1.0

def reward(a):
    return float(np.exp(-(a - 2.0) ** 2))

policy = LearnableGaussian(mu_init=0.0, log_sigma_init=0.5)
rng = np.random.default_rng(0)
lr, batch, n_steps = 0.02, 16, 800
running = 0.0; n_seen = 0
mus, sigmas, rs_avg = [], [], []

for step in range(n_steps):
    actions = np.array([policy.sample(rng) for _ in range(batch)])
    rs = np.array([reward(a) for a in actions])
    for r in rs:
        n_seen += 1; running += (r - running) / n_seen
    advs = rs - running
    g_mu = sum(A * policy.grads(a)[0] for a, A in zip(actions, advs)) / batch
    g_ls = sum(A * policy.grads(a)[1] for a, A in zip(actions, advs)) / batch
    policy.mu += lr * g_mu
    policy.log_sigma += lr * g_ls
    policy.log_sigma = float(np.clip(policy.log_sigma, -3.0, 2.0))
    mus.append(policy.mu); sigmas.append(policy.sigma); rs_avg.append(rs.mean())

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(mus, label="$\\mu$"); axes[0].axhline(2.0, color="r", ls="--", alpha=0.5, label="optimum")
ax2 = axes[0].twinx(); ax2.plot(sigmas, color="C1", label="$\\sigma$")
axes[0].set_xlabel("step"); axes[0].legend(loc="upper left"); ax2.legend(loc="lower right")
axes[0].set_title("Policy parameters")
axes[1].plot(rs_avg); axes[1].set_xlabel("step"); axes[1].set_ylabel("batch avg reward")
axes[1].set_title("Average reward per batch"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"Final: mu={policy.mu:.3f}, sigma={policy.sigma:.3f}")

$\mu$ should track from 0 toward 2; $\sigma$ should *contract* from ~1.65 to
~0.2 as the policy discovers the peak. The variance is itself an optimized
parameter — the policy automatically tightens around the optimum once it
finds one.

### Try this
- Change `reward` to a multimodal function:
  `lambda a: np.exp(-(a-2)**2) + 0.6 * np.exp(-(a+1)**2)`.
  Does the policy find the bigger peak? Or get stuck on the smaller one?
  Try several seeds.
- Set `log_sigma_init = -1.0` (start with tiny variance). The policy is
  initially almost deterministic at $\mu=0$. Does it ever discover the peak
  at 2? *(Usually no — exploration is locked in.)*
- Set `lr = 0.5`. How dramatic is the collapse?

## What's next

You've now seen the entire core of REINFORCE — log-derivative trick,
softmax/Gaussian policies, baselines, continuous action spaces — in
working code.

The two big problems we left unsolved:

1. **The baseline still wasn't learned.** Running mean and oracle are
   crude. The natural fix is to *learn* the baseline online via TD —
   that's the actor-critic family (Post 2b).
2. **REINFORCE breaks at high learning rates.** We saw a hint of this
   when you set `lr=2.0` above. The fix is to constrain how much the
   policy can change per update — that's PPO (Post 2c).

Then RLHF and DPO (Post 2d) are these same gradients, with rewards coming
from human preferences. The math is one continuous derivation from the
log-derivative trick you visualized at the top of this notebook.

### Suggested next steps
- Walk through [Post 2b](../posts/02b-actor-critic.qmd) and then build a
  similar notebook for actor-critic, swapping the running-mean baseline
  for a learned $V_\phi$.
- Modify `nano_agents.policy_gradient.train_reinforce` to support
  decayed learning rates. Does it help on harder problems?
- Read [the original 1992 Williams paper](https://link.springer.com/article/10.1007/BF00992696)
  if you want to see how this material was originally framed.